# Text-aware DTD — shared-backbone OCR head + proposal losses

One image encode, two heads. A **text head reads the Visual Perception Head (VPH) features** and
predicts the text mask *inside* the network; that mask is fused into the decoder. **No second RGB
backbone and no external OCR at inference** — a pretrained OCR is used only to supervise the head
during training, so limited data is not a constraint.

**Protocol:** train/val from `DocTamperV1-TrainingSet` (2000 / 200), test on
`DocTamperV1-TestingSet` (200). Disjoint manifests, seed 42.

**Loss** (proposal §3): `L = CE + λ_dice·Dice + λ_bound·Boundary` for tampering,
plus `λ_text·BCE` for the text head. The λ's are ablated.

**Controls learned the hard way** (each fixed a real artefact):
1. DTD backbone frozen — it already trained on TrainingSet, so fine-tuning it degrades it.
2. Backbone kept in `eval()` — `requires_grad=False` does *not* freeze BatchNorm running stats.
3. Fusion identity-initialised (zeroed final BN + linear merge) — step 0 equals the frozen checkpoint.
4. Val-based selection — an arm can never end below the baseline.

In [ ]:
# --- CONFIG ---
from pathlib import Path

CONFIG = {
    'PROJECT_REPO_URL': 'https://github.com/SamiraAbedini/HLCV-Project.git',
    'PROJECT_BRANCH': 'main',
    'PROJECT_DIR': '/content/HLCV-Project',
    'DOCTAMPER_DIR': '/content/DocTamper',

    'CHECKPOINT_DIR': '/content/drive/MyDrive/HLCV/checkpoints',       # shared (read)
    'OUT_ROOT':       '/content/drive/MyDrive/HLCV_samira/ta_design',  # yours (write)
    'MANIFEST_DIR':   '/content/drive/MyDrive/HLCV_samira/manifests/ta_train2000_val200_test200_seed42',
    'DRIVE_ZIP':      '/content/drive/MyDrive/HLCV_samira/data/doctamper.zip',
    'KAGGLE_DATASET': 'dinmkeljiame/doctamper',
    'DATA_ROOT':      '/content/doctamper_train_test',

    'SEED': 42,
    'TRAIN_SOURCE': 'DocTamperV1-TrainingSet',
    'TEST_SOURCE':  'DocTamperV1-TestingSet',
    'TRAIN_SIZE': 2000, 'VAL_SIZE': 200, 'TEST_SIZE': 200,

    'BATCH_SIZE': 2, 'TRAIN_STEPS': 600, 'ADAPTER_LR': 1e-4, 'WEIGHT_DECAY': 1e-2,
    'EVAL_EVERY': 100, 'PATIENCE': 3,
    'JPEG_QUALITY': 75, 'EVAL_THRESHOLD': 0.5, 'IMG': 512,
    'LAMBDA_TEXT': 0.5,                 # weight of the text-head BCE
    'TEXT_FEAT_IDX': 0,                 # which VPH feature the text head reads

    # Pretrained OCR used ONLY to supervise the text head (training) and as the reference
    # detector when measuring the head's precision/recall.
    'OCR_BACKEND': 'easyocr', 'OCR_DILATION': 2, 'OCR_CONFIDENCE_THRESHOLD': 0.0,
    'INIT_CHECKPOINT': 'dtd_doctamper.pth',
}
CONFIG

In [ ]:
# --- Setup ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
assert Path(CONFIG['CHECKPOINT_DIR']).exists(), (
    f"Not found: {CONFIG['CHECKPOINT_DIR']} -- add the shared HLCV folder shortcut to My Drive.")

%cd /content
if not Path(CONFIG['PROJECT_DIR']).exists():
    subprocess.run(['git', 'clone', '-q', '-b', CONFIG['PROJECT_BRANCH'],
                    CONFIG['PROJECT_REPO_URL'], CONFIG['PROJECT_DIR']], check=True)
if not Path(CONFIG['DOCTAMPER_DIR']).exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/qcf-568/DocTamper.git',
                    CONFIG['DOCTAMPER_DIR']], check=True)

!pip -q install lmdb six albumentations timm==0.4.12 segmentation_models_pytorch==0.2.1 easyocr pytesseract kaggle scikit-learn matplotlib
!pip -q install efficientnet_pytorch==0.7.1
!apt-get -qq install -y tesseract-ocr libjpeg-dev > /dev/null
!pip -q uninstall -y jpegio
!rm -rf /content/jpegio && git clone -q https://github.com/dwgoon/jpegio.git /content/jpegio
%cd /content/jpegio
!pip -q install .
%cd /content/DocTamper/models

sys.path.insert(0, CONFIG['PROJECT_DIR'])
import jpegio
print('jpegio.read:', hasattr(jpegio, 'read'))

In [ ]:
# --- Data (cached zip -> temp disk) + checkpoints + manifests ---
import getpass, shutil
DATA_ROOT = Path(CONFIG['DATA_ROOT']); DATA_ROOT.mkdir(parents=True, exist_ok=True)
need = [DATA_ROOT / CONFIG['TRAIN_SOURCE'] / 'data.mdb', DATA_ROOT / CONFIG['TEST_SOURCE'] / 'data.mdb']

if not all(p.exists() for p in need):
    zp = Path('/content/doctamper_kaggle/doctamper.zip')
    if not zp.exists():
        zp = Path(CONFIG['DRIVE_ZIP'])
    if not zp.exists():
        kdir = Path.home() / '.kaggle'; kdir.mkdir(parents=True, exist_ok=True)
        if not (kdir / 'access_token').exists() and not (kdir / 'kaggle.json').exists():
            (kdir / 'access_token').write_text(getpass.getpass('Paste Kaggle API token: ').strip())
            (kdir / 'access_token').chmod(0o600)
        zp = Path('/content/doctamper_kaggle/doctamper.zip'); zp.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['kaggle', 'datasets', 'download', '-d', CONFIG['KAGGLE_DATASET'],
                        '-p', str(zp.parent)], check=True)
    for pat in [f"{CONFIG['TRAIN_SOURCE']}/*", f"{CONFIG['TEST_SOURCE']}/*"]:
        print('extracting', pat)
        subprocess.run(['unzip', '-o', '-q', str(zp), pat, '-d', str(DATA_ROOT)], check=True)
for p in need:
    assert p.exists(), f'Missing {p}'

%cd /content/DocTamper/models
if not Path('qt_table.pk').exists():
    shutil.copy2(Path(CONFIG['DOCTAMPER_DIR']) / 'qt_table.pk', 'qt_table.pk')
for f in ['vph_imagenet.pt', 'swin_imagenet.pt', CONFIG['INIT_CHECKPOINT']]:
    if not Path(f).exists():
        srcp = Path(CONFIG['CHECKPOINT_DIR']) / f
        assert srcp.exists(), f'Missing {srcp}'
        shutil.copy2(srcp, f); print('staged', f)

MAN = Path(CONFIG['MANIFEST_DIR'])
if not (MAN / 'test.json').exists():
    cmd = [sys.executable, f"{CONFIG['PROJECT_DIR']}/scripts/generate_doctamper_subset.py",
           '--data-root', str(DATA_ROOT), '--output-dir', str(MAN), '--seed', str(CONFIG['SEED']),
           '--train-source', CONFIG['TRAIN_SOURCE'], '--test-source', CONFIG['TEST_SOURCE'],
           '--train-size', str(CONFIG['TRAIN_SIZE']), '--val-size', str(CONFIG['VAL_SIZE']),
           '--test-size', str(CONFIG['TEST_SIZE'])]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(r.stderr)
for s in ['train', 'val', 'test']:
    assert (MAN / f'{s}.json').exists()
print('manifests ready:', MAN)

In [ ]:
# --- Model, losses, loaders ---
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, json, copy, time
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score

_ORIG_LOAD = torch.load
torch.load = lambda *a, **k: _ORIG_LOAD(*a, **{**k, 'weights_only': False})
from dtd import *                                     # pickled backbone classes into __main__
from src.losses import CombinedTamperLoss
from src.fusion import TextPriorFusion
from src.doctamper_dataset import ManifestDocTamperDataset
from src.ocr_backends import OCRConfig, create_ocr_backend
from src.ocr_cache import OCRDetectionCache
from src.ocr_eval import OCRCoverageMeter

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(CONFIG['SEED']); np.random.seed(CONFIG['SEED'])
OUT = Path(CONFIG['OUT_ROOT']); OUT.mkdir(parents=True, exist_ok=True)
IMG = CONFIG['IMG']

def make_loader(split, shuffle):
    ds = ManifestDocTamperDataset(DATA_ROOT, MAN / f'{split}.json', 'qt_table.pk', CONFIG['JPEG_QUALITY'])
    return DataLoader(ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=shuffle,
                      num_workers=0, drop_last=shuffle)
train_loader, val_loader, test_loader = make_loader('train', True), make_loader('val', False), make_loader('test', False)
print(f'train {len(train_loader.dataset)} | val {len(val_loader.dataset)} | test {len(test_loader.dataset)}')

def build_model():
    m = seg_dtd('', 2).to(DEVICE)
    for mod in m.modules():                            # old pickles predate nn.GELU.approximate
        if isinstance(mod, nn.GELU) and not hasattr(mod, 'approximate'):
            mod.approximate = 'none'
    sd = torch.load(CONFIG['INIT_CHECKPOINT'], map_location='cpu')['state_dict']
    m.load_state_dict({k.replace('module.', ''): v for k, v in sd.items()}, strict=False)
    return m

def forward_dtd(model, batch):
    return model(batch['image'].to(DEVICE), batch['rgb'].to(DEVICE), batch['q'].unsqueeze(1).to(DEVICE))

MEAN = np.array([0.485, 0.455, 0.406]); STD = np.array([0.229, 0.224, 0.225])
def denorm(t):
    a = t.permute(1, 2, 0).cpu().numpy() * STD + MEAN
    return np.clip(a * 255, 0, 255).astype(np.uint8)
print('model helpers ready | device', DEVICE)

In [ ]:
# --- Pretrained OCR: supervision for the text head (training) + reference detector (eval) ---
# This is the ONLY place a pretrained OCR is used. It never runs at inference.
ocr_cfg = OCRConfig(backend=CONFIG['OCR_BACKEND'], languages=('en',),
                    confidence_threshold=CONFIG['OCR_CONFIDENCE_THRESHOLD'],
                    dilation=CONFIG['OCR_DILATION'], easyocr_gpu=(DEVICE == 'cuda'))
ocr_backend = create_ocr_backend(ocr_cfg)
ocr_cache = OCRDetectionCache(OUT / 'ocr_cache', ocr_cfg)

def ocr_masks(batch):
    """Pretrained-OCR text masks for a batch -> (B,1,H,W) float on device (cached per sample)."""
    out = []
    for t, sid in zip(batch['image'], batch['sample_id']):
        img = denorm(t)
        det, _ = ocr_cache.get_or_compute(sid, img, ocr_backend)
        out.append(ocr_cache.mask_from_detections(det, img.shape))
    return torch.from_numpy(np.stack(out)[:, None]).float().to(DEVICE)

t0 = time.time()
_ = ocr_masks(next(iter(train_loader)))
print(f'OCR pseudo-labeller ready ({CONFIG["OCR_BACKEND"]}), first batch {time.time()-t0:.1f}s')

In [ ]:
# --- ARCHITECTURE (TA's design): text head on shared VPH features + decoder fusion ---
class TextHead(nn.Module):
    """Lightweight text-detection head on a shared VPH feature map -> 1-channel text logits."""
    def __init__(self, in_ch, mid=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, mid, 3, padding=1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(True),
            nn.Conv2d(mid, mid, 3, padding=1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(True),
            nn.Conv2d(mid, 1, 1))
    def forward(self, x): return self.net(x)

class HeadWithPrior(nn.Module):
    """Fuse the predicted text mask into the final decoder feature, then run the seg head."""
    def __init__(self, head, in_ch):
        super().__init__()
        self.fusion = TextPriorFusion(in_ch); self.head = head; self.text_mask = None
    def forward(self, feat):
        return self.head(self.fusion(feat, self.text_mask))

def probe_vph(model):
    """Report the VPH output feature shapes (one forward, no grad)."""
    grabbed = {}
    h = model.model.vph.register_forward_hook(lambda m, i, o: grabbed.update(o=o))
    with torch.no_grad():
        _ = forward_dtd(model, next(iter(val_loader)))
    h.remove()
    feats = list(grabbed['o']) if isinstance(grabbed['o'], (list, tuple)) else [grabbed['o']]
    for i, f in enumerate(feats):
        print(f'  vph feat[{i}] channels={f.shape[1]} spatial={tuple(f.shape[-2:])}')
    return feats

def wire_ocr_head(model, feat_idx, c_vph):
    """One backbone, two heads. A forward hook on vph computes the text mask from the SHARED
    features and feeds it to the decoder fusion -- no second RGB backbone, no OCR at inference."""
    core = model.model
    if hasattr(core, '_ta_hook'):
        core._ta_hook.remove()
    core.text_head = TextHead(c_vph).to(DEVICE)

    head = core.segmentation_head
    while hasattr(head, 'head'):
        head = head.head
    hp = HeadWithPrior(head, head[0].in_channels).to(DEVICE)
    nn.init.zeros_(hp.fusion.block[-1].weight)         # identity init: block(...) == 0
    nn.init.zeros_(hp.fusion.block[-1].bias)
    hp.fusion.act = nn.Identity()                      # linear merge so identity is reachable
    core.segmentation_head = hp

    def hook(m, i, o):
        feats = list(o) if isinstance(o, (list, tuple)) else [o]
        t = core.text_head(feats[feat_idx])
        t = F.interpolate(t, size=(IMG, IMG), mode='bilinear', align_corners=False)
        core._text_logits = t
        core.segmentation_head.text_mask = torch.sigmoid(t)
    core._ta_hook = core.vph.register_forward_hook(hook)
    return core

def adapter_params(model):
    """Freeze the pretrained DTD; train only the two NEW modules."""
    core = model.model
    for p in model.parameters():
        p.requires_grad_(False)
    for p in core.text_head.parameters():
        p.requires_grad_(True)
    for p in core.segmentation_head.fusion.parameters():
        p.requires_grad_(True)
    return [p for p in model.parameters() if p.requires_grad]

def set_train_mode(model):
    """Backbone eval() so its BatchNorm running stats stay frozen; new modules train()."""
    model.eval()
    model.model.text_head.train()
    model.model.segmentation_head.fusion.train()

print('architecture ready')

In [ ]:
# --- evaluate() and the joint-loss training loop ---
@torch.no_grad()
def evaluate(model, loader, keep_frac=0.05, seed=0):
    """No external OCR: the text head supplies the mask via its hook (or there is no prior)."""
    model.eval(); rng = np.random.default_rng(seed)
    tp = fp = fn = 0; probs, labels = [], []
    for batch in loader:
        target = batch['label'].squeeze(1).long().to(DEVICE)
        with autocast(enabled=(DEVICE == 'cuda')):
            logits = forward_dtd(model, batch)
            if logits.shape[-2:] != target.shape[-2:]:
                logits = F.interpolate(logits, size=target.shape[-2:], mode='bilinear', align_corners=False)
        prob = torch.softmax(logits.float(), 1)[:, 1]
        pred, gt = prob > CONFIG['EVAL_THRESHOLD'], target.bool()
        tp += (pred & gt).sum().item(); fp += (pred & ~gt).sum().item(); fn += (~pred & gt).sum().item()
        p = prob.flatten().cpu().numpy(); l = gt.flatten().cpu().numpy()
        idx = rng.choice(len(p), max(1, int(len(p) * keep_frac)), replace=False)
        probs.append(p[idx].astype(np.float32)); labels.append(l[idx].astype(np.uint8))
    prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
    y, s = np.concatenate(labels), np.concatenate(probs)
    return ({'pixel_f1': 2 * prec * rec / (prec + rec + 1e-9), 'precision': prec, 'recall': rec,
             'iou': tp / (tp + fp + fn + 1e-9),
             'auc': float(roc_auc_score(y, s)) if y.min() != y.max() else float('nan')}, y, s)

bce = nn.BCEWithLogitsLoss()

def train_adapter(model, criterion, steps, tag=''):
    """Joint loss: tampering (CE + Dice + Boundary) + LAMBDA_TEXT * BCE(text head, OCR pseudo-label)."""
    core = model.model
    params = adapter_params(model)
    opt = torch.optim.AdamW(params, lr=CONFIG['ADAPTER_LR'], weight_decay=CONFIG['WEIGHT_DECAY'])
    scaler = GradScaler(); it = iter(train_loader)

    m0, _, _ = evaluate(model, val_loader)                      # identity init => frozen checkpoint
    snap = lambda: {'text': copy.deepcopy(core.text_head.state_dict()),
                    'fus': copy.deepcopy(core.segmentation_head.fusion.state_dict())}
    best = {'f1': m0['pixel_f1'], 'step': 0, 'state': snap()}
    print(f"  [{tag}] trainable {sum(p.numel() for p in params):,} | val@0000 f1={m0['pixel_f1']:.4f}")
    stale = 0
    set_train_mode(model)
    for step in range(steps):
        try: batch = next(it)
        except StopIteration: it = iter(train_loader); batch = next(it)
        pseudo = ocr_masks(batch)                                # pretrained-OCR supervision
        target = batch['label'].squeeze(1).long().to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with autocast(enabled=(DEVICE == 'cuda')):
            logits = forward_dtd(model, batch)                   # hook sets core._text_logits
            if logits.shape[-2:] != target.shape[-2:]:
                logits = F.interpolate(logits, size=target.shape[-2:], mode='bilinear', align_corners=False)
            tamper_loss, parts = criterion(logits, target)
            text_loss = bce(core._text_logits.float(), pseudo)
            loss = tamper_loss + CONFIG['LAMBDA_TEXT'] * text_loss
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()

        if (step + 1) % CONFIG['EVAL_EVERY'] == 0:
            m, _, _ = evaluate(model, val_loader)
            better = m['pixel_f1'] > best['f1']
            print(f"  [{tag}] val@{step+1:04d} f1={m['pixel_f1']:.4f} tamper={float(parts['total']):.4f} "
                  f"text={text_loss.item():.4f}{'  <- best' if better else ''}")
            if better:
                best = {'f1': m['pixel_f1'], 'step': step + 1, 'state': snap()}; stale = 0
            else:
                stale += 1
                if stale >= CONFIG['PATIENCE']:
                    print(f'  [{tag}] early stop at {step+1}'); break
            set_train_mode(model)
    core.text_head.load_state_dict(best['state']['text'])
    core.segmentation_head.fusion.load_state_dict(best['state']['fus'])
    print(f"  [{tag}] restored step {best['step']} (val f1={best['f1']:.4f})")
    return best

print('train/eval ready')

In [ ]:
# --- Arm 1: baseline = frozen DTD checkpoint (no OCR head, no training) ---
results, curves, selection = {}, {}, {}
model = build_model()
print('VPH feature shapes:')
feats = probe_vph(model)
C_VPH = feats[CONFIG['TEXT_FEAT_IDX']].shape[1]
print('text head will read feat[%d] with %d channels' % (CONFIG['TEXT_FEAT_IDX'], C_VPH))

results['baseline (frozen DTD)'], y, s = evaluate(model, test_loader)
curves['baseline (frozen DTD)'] = (y, s)
print('baseline:', {k: round(v, 4) for k, v in results['baseline (frozen DTD)'].items()})
del model; torch.cuda.empty_cache()

In [ ]:
# --- Arms 2-4: OCR head + loss ablation (proposal: CE, +Dice, +Boundary) ---
LOSS_CONFIGS = {
    '+OCR head, CE only':          (0.0, 0.0),
    '+OCR head, CE+Dice':          (1.0, 0.0),
    '+OCR head, CE+Dice+Boundary': (1.0, 0.5),
}
(OUT / 'arms').mkdir(parents=True, exist_ok=True)

for tag, (ld, lb) in LOSS_CONFIGS.items():
    print(f'=== {tag} (lambda_dice={ld}, lambda_bound={lb}) ===')
    model = build_model()
    wire_ocr_head(model, CONFIG['TEXT_FEAT_IDX'], C_VPH)
    crit = CombinedTamperLoss(lambda_dice=ld, lambda_bound=lb)
    best = train_adapter(model, crit, CONFIG['TRAIN_STEPS'], tag=tag.split(',')[-1].strip())
    selection[tag] = {'val_f1': best['f1'], 'step': best['step']}
    met, y, s = evaluate(model, test_loader)
    results[tag], curves[tag] = met, (y, s)
    torch.save({'text_head': model.model.text_head.state_dict(),
                'fusion': model.model.segmentation_head.fusion.state_dict()},
               OUT / 'arms' / f"{tag.replace(' ', '_').replace('+','').replace(',','')}.pth")
    print(' TEST:', {k: round(v, 4) for k, v in met.items()})
    del model; torch.cuda.empty_cache()

In [ ]:
# --- Threshold sweep (F1 at each arm's optimum, not just 0.5) ---
THRESHOLDS = np.linspace(0.05, 0.95, 19)
sweep = {}
for name, (y, s) in curves.items():
    f1s = []
    for t in THRESHOLDS:
        pred = s >= t
        tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum()); fn = int((~pred & (y == 1)).sum())
        p = tp / (tp + fp + 1e-9); r = tp / (tp + fn + 1e-9)
        f1s.append(2 * p * r / (p + r + 1e-9))
    f1s = np.array(f1s); b = int(f1s.argmax())
    sweep[name] = {'thresholds': THRESHOLDS, 'f1s': f1s}
    results[name]['f1_at_best_threshold'] = float(f1s[b])
    results[name]['best_threshold'] = float(THRESHOLDS[b])

print(f'{"arm":<30}{"F1@0.5":>9}{"F1@best":>10}{"best_t":>8}{"IoU":>9}{"AUC":>9}')
for k in results:
    v = results[k]
    print(f"{k:<30}{v['pixel_f1']:>9.4f}{v['f1_at_best_threshold']:>10.4f}"
          f"{v['best_threshold']:>8.2f}{v['iou']:>9.4f}{v['auc']:>9.4f}")

In [ ]:
# --- OCR MODULE EVALUATION (the TA's question) ------------------------------------------------
# Two complementary views, on the TEST split:
#  (a) DETECTION precision/recall of the learned text head, using the pretrained OCR as the
#      reference detector. DocTamper has no text-box ground truth, so a pretrained detector is
#      the only available reference -- this is the closest measurable analogue of "P/R on boxes".
#  (b) COVERAGE of tampered regions -- the quantity that actually decides downstream failure,
#      since a missed tampered region is unrecoverable (the high-recall region-proposal target).
BEST_ARM = max((k for k in results if k != 'baseline (frozen DTD)'),
               key=lambda k: results[k]['pixel_f1'])
print('evaluating the OCR head from:', BEST_ARM)

model = build_model(); wire_ocr_head(model, CONFIG['TEXT_FEAT_IDX'], C_VPH)
ck = torch.load(OUT / 'arms' / f"{BEST_ARM.replace(' ', '_').replace('+','').replace(',','')}.pth",
                map_location='cpu')
model.model.text_head.load_state_dict(ck['text_head'])
model.model.segmentation_head.fusion.load_state_dict(ck['fusion'])
model.eval()

det_tp = det_fp = det_fn = 0
head_meter, ocr_meter = OCRCoverageMeter(coverage_thresh=0.5), OCRCoverageMeter(coverage_thresh=0.5)
head_area, ocr_area = [], []
with torch.no_grad():
    for batch in test_loader:
        ref = ocr_masks(batch)[:, 0].cpu().numpy() > 0            # pretrained-OCR reference
        with autocast(enabled=(DEVICE == 'cuda')):
            _ = forward_dtd(model, batch)                          # hook fills _text_logits
        pred = (torch.sigmoid(model.model._text_logits.float())[:, 0].cpu().numpy() > 0.5)
        tamper = batch['label'][:, 0].numpy() > 0
        for b in range(pred.shape[0]):
            det_tp += int((pred[b] & ref[b]).sum())
            det_fp += int((pred[b] & ~ref[b]).sum())
            det_fn += int((~pred[b] & ref[b]).sum())
            head_area.append(float(pred[b].mean())); ocr_area.append(float(ref[b].mean()))
            if tamper[b].any():
                head_meter.update(pred[b], tamper[b])
                ocr_meter.update(ref[b], tamper[b])

det_p = det_tp / (det_tp + det_fp + 1e-9); det_r = det_tp / (det_tp + det_fn + 1e-9)
ocr_report = {
    'head_detection_precision_vs_pretrained_ocr': det_p,
    'head_detection_recall_vs_pretrained_ocr': det_r,
    'head_detection_f1_vs_pretrained_ocr': 2 * det_p * det_r / (det_p + det_r + 1e-9),
    'head_tamper_coverage': head_meter.compute()['pixel_recall_coverage'],
    'head_component_recall': head_meter.compute()['component_recall'],
    'head_text_area_ratio': float(np.mean(head_area)),
    'pretrained_ocr_tamper_coverage': ocr_meter.compute()['pixel_recall_coverage'],
    'pretrained_ocr_component_recall': ocr_meter.compute()['component_recall'],
    'pretrained_ocr_text_area_ratio': float(np.mean(ocr_area)),
}
for k, v in ocr_report.items():
    print(f'{k:<45} {v:.4f}')
with (OUT / 'ocr_module_report.json').open('w') as f:
    json.dump(ocr_report, f, indent=2, sort_keys=True)
del model; torch.cuda.empty_cache()

In [ ]:
# --- Small-region breakdown (the proposal's core motivation) ---
import cv2
BUCKETS = [(0, 200), (200, 1000), (1000, 5000), (5000, 10**9)]
NAMES = ['<200', '200-1k', '1k-5k', '>5k']

@torch.no_grad()
def region_breakdown(model):
    model.eval(); rec = [0.0] * 4; cnt = [0] * 4
    for batch in test_loader:
        target = batch['label'][:, 0].numpy() > 0
        with autocast(enabled=(DEVICE == 'cuda')):
            logits = forward_dtd(model, batch)
            if logits.shape[-2:] != batch['label'].shape[-2:]:
                logits = F.interpolate(logits, size=batch['label'].shape[-2:], mode='bilinear', align_corners=False)
        pred = (torch.softmax(logits.float(), 1)[:, 1] > CONFIG['EVAL_THRESHOLD']).cpu().numpy()
        for b in range(pred.shape[0]):
            num, lab = cv2.connectedComponents(target[b].astype(np.uint8), connectivity=8)
            for c in range(1, num):
                comp = lab == c; area = int(comp.sum())
                r = float((comp & pred[b]).sum()) / max(1, area)
                for k, (lo, hi) in enumerate(BUCKETS):
                    if lo <= area < hi:
                        rec[k] += r; cnt[k] += 1; break
    return [(cnt[k], rec[k] / cnt[k] if cnt[k] else float('nan')) for k in range(4)]

m = build_model(); base_bd = region_breakdown(m); del m; torch.cuda.empty_cache()
m = build_model(); wire_ocr_head(m, CONFIG['TEXT_FEAT_IDX'], C_VPH)
ck = torch.load(OUT / 'arms' / f"{BEST_ARM.replace(' ', '_').replace('+','').replace(',','')}.pth", map_location='cpu')
m.model.text_head.load_state_dict(ck['text_head']); m.model.segmentation_head.fusion.load_state_dict(ck['fusion'])
ocr_bd = region_breakdown(m); del m; torch.cuda.empty_cache()

print(f'{"size":>8}{"n":>6}{"baseline":>11}{"+OCR head":>12}')
for nm, (n, rb), (_, ro) in zip(NAMES, base_bd, ocr_bd):
    print(f'{nm:>8}{n:>6}{rb:>11.4f}{ro:>12.4f}')

In [ ]:
# --- Save tables + figures ---
import matplotlib as mpl, matplotlib.pyplot as plt
SURFACE, INK, INK2, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#e3e3e0'
SERIES = ['#2a78d6', '#008300', '#e87ba4', '#eda100']
FIGS = OUT / 'figures'; FIGS.mkdir(parents=True, exist_ok=True)
mpl.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
                     'text.color': INK, 'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2,
                     'axes.edgecolor': GRID, 'axes.linewidth': .8, 'font.size': 11, 'axes.titlesize': 13,
                     'axes.titleweight': 'semibold', 'axes.titlelocation': 'left', 'legend.frameon': False,
                     'figure.dpi': 140, 'savefig.dpi': 220, 'savefig.bbox': 'tight'})
def style(ax, ylabel=None, ymax=1.0):
    for sp in ('top', 'right', 'left'): ax.spines[sp].set_visible(False)
    ax.yaxis.grid(True, color=GRID, linewidth=.8); ax.set_axisbelow(True); ax.set_ylim(0, ymax)
    if ylabel: ax.set_ylabel(ylabel)

COLS = ['pixel_f1', 'f1_at_best_threshold', 'precision', 'recall', 'iou', 'auc']
lines = ['# Results — TrainingSet(2000/200) -> TestingSet(200), seed 42', '',
         '| arm | ' + ' | '.join(COLS) + ' |', '|' + '---|' * (len(COLS) + 1)]
for k, v in results.items():
    lines.append(f'| {k} | ' + ' | '.join(f'{v[c]:.4f}' for c in COLS) + ' |')
lines += ['', '## OCR module (test split)', '']
lines += [f'- {k}: {v:.4f}' for k, v in ocr_report.items()]
lines += ['', '## Recall by tampered-region size', '', '| size | n | baseline | +OCR head |', '|---|---|---|---|']
for nm, (n, rb), (_, ro) in zip(NAMES, base_bd, ocr_bd):
    lines.append(f'| {nm} | {n} | {rb:.4f} | {ro:.4f} |')
(OUT / 'results.md').write_text('\n'.join(lines) + '\n')

arms = list(results.keys())
short = [a.replace('+OCR head, ', '') for a in arms]
fig, ax = plt.subplots(figsize=(8, 4.2))
x = np.arange(len(arms)); w = 0.38
for i, (lab, key) in enumerate([('F1 @ 0.5', 'pixel_f1'), ('F1 @ best t', 'f1_at_best_threshold')]):
    vals = [results[a][key] for a in arms]
    b = ax.bar(x + (i - .5) * w, vals, width=w * .92, color=SERIES[i], label=lab, zorder=3)
    for r, v in zip(b, vals):
        ax.text(r.get_x() + r.get_width() / 2, v + .015, f'{v:.3f}', ha='center', fontsize=9, color=INK2)
ax.set_xticks(x); ax.set_xticklabels(short, rotation=12, ha='right'); ax.legend(ncol=2)
style(ax, 'Pixel-F1'); ax.set_title('Shared-backbone OCR head + loss ablation')
for e in ('png', 'svg'): fig.savefig(FIGS / f'fig1_results.{e}')
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
xb = np.arange(len(NAMES))
for i, (lab, bd) in enumerate([('baseline', base_bd), ('+OCR head', ocr_bd)]):
    vals = [v for _, v in bd]
    b = ax.bar(xb + (i - .5) * w, vals, width=w * .92, color=SERIES[i], label=lab, zorder=3)
    for r, v in zip(b, vals):
        if np.isfinite(v): ax.text(r.get_x() + r.get_width() / 2, v + .015, f'{v:.2f}', ha='center', fontsize=9, color=INK2)
ax.set_xticks(xb); ax.set_xticklabels([f'{n}\n(n={c})' for n, (c, _) in zip(NAMES, base_bd)])
ax.legend(ncol=2); style(ax, 'component recall'); ax.set_title('Recall by tampered-region size')
for e in ('png', 'svg'): fig.savefig(FIGS / f'fig2_small_regions.{e}')
plt.show()
print('saved ->', OUT)

In [ ]:
# --- Qualitative: image | learned TEXT mask | pretrained OCR | GT tamper | prediction | missed ---
model = build_model(); wire_ocr_head(model, CONFIG['TEXT_FEAT_IDX'], C_VPH)
ck = torch.load(OUT / 'arms' / f"{BEST_ARM.replace(' ', '_').replace('+','').replace(',','')}.pth", map_location='cpu')
model.model.text_head.load_state_dict(ck['text_head'])
model.model.segmentation_head.fusion.load_state_dict(ck['fusion'])
model.eval()

batch = next(iter(test_loader))
with torch.no_grad():
    ref = ocr_masks(batch)[:, 0].cpu().numpy() > 0
    with autocast(enabled=(DEVICE == 'cuda')):
        logits = forward_dtd(model, batch)
        if logits.shape[-2:] != batch['label'].shape[-2:]:
            logits = F.interpolate(logits, size=batch['label'].shape[-2:], mode='bilinear', align_corners=False)
    txt = torch.sigmoid(model.model._text_logits.float())[:, 0].cpu().numpy()
    prob = torch.softmax(logits.float(), 1)[:, 1].cpu().numpy()

n = min(3, len(prob))
fig, axes = plt.subplots(n, 6, figsize=(18, 3.1 * n)); axes = np.atleast_2d(axes)
for i in range(n):
    img = denorm(batch['image'][i]); gt = batch['label'][i, 0].numpy() > 0
    missed = gt & ~(txt[i] > 0.5)
    panels = [(img, 'image', None), (txt[i], 'text head (learned)', 'gray'),
              (ref[i], 'pretrained OCR (ref)', 'gray'), (gt, 'GT tamper', 'gray'),
              (prob[i], 'predicted tamper', 'magma')]
    for j, (im, t, cm) in enumerate(panels):
        axes[i, j].imshow(im, cmap=cm, vmin=0 if cm else None, vmax=1 if cm else None)
        axes[i, j].set_title(t, fontsize=10, loc='left'); axes[i, j].axis('off')
    ov = img.copy(); ov[missed] = np.array([227, 73, 72], np.uint8)
    axes[i, 5].imshow(ov); axes[i, 5].set_title('tamper missed by text head', fontsize=10, loc='left')
    axes[i, 5].axis('off')
fig.tight_layout()
for e in ('png', 'svg'): fig.savefig(FIGS / f'fig3_qualitative.{e}')
plt.show()

print('\nFILES:')
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.suffix in {'.md', '.json', '.png', '.svg', '.pth'}:
        print('  ', p.relative_to(OUT))